# GraphRAG

Extract a knowledge graph from the documents and answer questions by walking it.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

**Extra requirements:** `pip install networkx`

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import networkx as nx

G = nx.MultiDiGraph()
for i, chunk in enumerate(chunks[:15]):
    for s, r, o in extract_triples(chunk):
        G.add_edge(s, o, relation=r, chunk=i)
print(G.number_of_nodes(), "entities,", G.number_of_edges(), "relations")
list(G.edges(data="relation"))[:8]

## Query: find entities in the question, then expand their neighbourhood

In [ ]:
def graph_context(query, hops=1, max_facts=15):
    entities = llm_json(f"List the key entities in this question as a JSON list of lowercase strings.\n\n{query}") or []
    nodes = [n for n in G.nodes if any(e.lower() in n or n in e.lower() for e in entities)]
    facts = set()
    frontier = set(nodes)
    for _ in range(hops):
        nxt = set()
        for n in frontier:
            for u, v, d in list(G.out_edges(n, data=True)) + list(G.in_edges(n, data=True)):
                facts.add(f"{u} {d['relation']} {v}")
                nxt.update([u, v])
        frontier = nxt
    return nodes, sorted(facts)[:max_facts]

query = "What does HNSW do and which libraries use vector indexes?"
nodes, facts = graph_context(query)
print("matched:", nodes); print("\n".join(facts))
print(answer(query, facts) if facts else "no graph facts matched")

Graph retrieval shines for multi-hop questions (A relates to B relates to C) that no single chunk answers.